# 04 — Modeling prep & spatial split

Build X/y, make a spatial-block group when no natural one exists, do a group-aware split, and sanity-check for leakage and distribution drift.

In [ ]:
from geo_daily_tools.sample_data import messy_sensor_records
from geo_daily_tools.validation import validate_sensor_records
from geo_daily_tools.modeling import (
    feature_missingness_report,
    prepare_model_inputs,
    coordinate_grid_block,
    group_train_test_split,
    check_group_leakage,
    compare_distributions,
    simple_group_holdout,
)

valid_df, _ = validate_sensor_records(messy_sensor_records())
valid_df

## Feature missingness

In [ ]:
feature_missingness_report(valid_df, ['lat', 'lon'])

## Build X / y

In [ ]:
X, y, model_df = prepare_model_inputs(valid_df, ['lat', 'lon'], 'reading')
X

## Build a spatial-block group when no natural one exists

Floor-buckets lat/lon into a coarse grid you can use as `group_col` for spatial CV.

In [ ]:
blocked = coordinate_grid_block(valid_df, size_deg=0.05)
blocked[['obs_id', 'lat', 'lon', 'grid_block']]

## Group-aware train/test split

Random row splits leak spatial structure between train and test. Splitting by tile/site/scene gives an honest validation set.

In [ ]:
train, test = group_train_test_split(valid_df, group_col='tile_id', test_size=0.5)
check_group_leakage(train, test, 'tile_id')

## Distribution sanity check

In [ ]:
compare_distributions(train, test, ['lat', 'lon', 'reading'])

## Hold out specific groups by name

In [ ]:
train2, test2 = simple_group_holdout(valid_df, 'tile_id', ['tile_001'])
print('train tiles:', train2['tile_id'].unique().tolist())
print('test tiles: ', test2['tile_id'].unique().tolist())